# 🧠 Brain Tumor MRI Classification — Model Training

**Architecture:** Custom CNN + Transfer Learning (EfficientNetB0)  
**Framework:** TensorFlow / Keras  
**Strategy:** Train custom CNN → Fine-tune EfficientNet → Compare


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ─── Config ──────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 30
NUM_CLASSES = 4
SEED        = 42

DATA_DIR  = Path('../data')
TRAIN_DIR = str(DATA_DIR / 'Training')
TEST_DIR  = str(DATA_DIR / 'Testing')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
print('Config set ✅')

In [ ]:
# ─── Data Augmentation & Generators ──────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'\nClass indices: {train_gen.class_indices}')

In [ ]:
# ─── Compute Class Weights (handle imbalance) ─────────────────────
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights = dict(enumerate(class_weights_arr))
print('Class weights:', {CLASSES[k]: round(v, 3) for k, v in class_weights.items()})

In [ ]:
# ─── Model 1: Custom CNN ──────────────────────────────────────────
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.3),

        # Block 4
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        # Dense Head
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='BrainTumorCNN')
    return model

cnn_model = build_custom_cnn()
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

In [ ]:
# ─── Model 2: Transfer Learning — EfficientNetB0 ─────────────────
def build_efficientnet(input_shape=(224, 224, 3), num_classes=4):
    base = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    # Freeze base initially
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB0_BrainTumor')
    return model, base

eff_model, base_model = build_efficientnet()
eff_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f'EfficientNetB0 total params: {eff_model.count_params():,}')

In [ ]:
# ─── Callbacks ────────────────────────────────────────────────────
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=str(MODEL_DIR / f'{model_name}_best.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

In [ ]:
# ─── Train Custom CNN ─────────────────────────────────────────────
print('=' * 50)
print('Training Custom CNN...')
print('=' * 50)

cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks('custom_cnn'),
    verbose=1
)

In [ ]:
# ─── Train EfficientNet (Phase 1: frozen base) ────────────────────
print('=' * 50)
print('Training EfficientNetB0 — Phase 1 (frozen base)...')
print('=' * 50)

eff_history = eff_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=get_callbacks('efficientnet_phase1'),
    verbose=1
)

In [ ]:
# ─── Fine-tune EfficientNet (Phase 2: unfreeze top layers) ────────
# Unfreeze top 30 layers for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Very low LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Trainable layers: {sum(1 for l in eff_model.layers if l.trainable)}')
print('=' * 50)
print('Fine-tuning EfficientNetB0 — Phase 2...')
print('=' * 50)

eff_finetune_history = eff_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=get_callbacks('efficientnet_finetuned'),
    verbose=1
)

In [ ]:
# ─── Plot Training Curves ─────────────────────────────────────────
def plot_history(history, title, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    colors = ['#58a6ff', '#f78166']

    # Accuracy
    axes[0].plot(history.history['accuracy'], color=colors[0], label='Train', lw=2)
    axes[0].plot(history.history['val_accuracy'], color=colors[1], label='Val', lw=2)
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'], color=colors[0], label='Train', lw=2)
    axes[1].plot(history.history['val_loss'], color=colors[1], label='Val', lw=2)
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_history(cnn_history, 'Custom CNN — Training Curves', '../results/cnn_training_curves.png')
plot_history(eff_history, 'EfficientNetB0 Phase 1 — Training Curves', '../results/eff_phase1_curves.png')

In [ ]:
# ─── Save Final Models ─────────────────────────────────────────────
cnn_model.save(str(MODEL_DIR / 'custom_cnn_final.keras'))
eff_model.save(str(MODEL_DIR / 'efficientnet_finetuned.keras'))
print('✅ Models saved to models/')